# NIH ChestX-ray14 + RAG — Google Colab Runner
Run cells top to bottom. Tuned for the **free T4** tier: patient-wise subsample,
checkpoints saved to Google Drive so a disconnect won't lose your work.

**Before you start:** Runtime ▸ Change runtime type ▸ **T4 GPU**.

## 1. Confirm the GPU (do this first — if no GPU, retry later)

In [ ]:
!nvidia-smi
import torch
print('GPU visible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (MB):', torch.cuda.get_device_properties(0).total_memory // 1024**2)

## 2. Get the project code
Either clone from your GitHub (after you push it) **or** upload the zip via the
Files panel and unzip. Pick one.

In [ ]:
# Option A — clone from GitHub (recommended once pushed):
# !git clone https://github.com/G1rixh/<your-repo>.git
# %cd <your-repo>

# Option B — upload nih-chestxray14-rag.zip via the Files panel, then:
!unzip -q nih-chestxray14-rag.zip
%cd nih-chestxray14-rag
!ls

## 3. Install dependencies

In [ ]:
!pip install -q scikit-learn pandas matplotlib sentence-transformers faiss-cpu \
    google-generativeai mlflow
# torch/torchvision are preinstalled on Colab with CUDA.

## 4. Download the dataset via the Kaggle API
Get `kaggle.json` from kaggle.com ▸ Account ▸ Create New API Token, then upload it
when prompted below.

In [ ]:
from google.colab import files
import os, json
print('Upload your kaggle.json:')
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip install -q kaggle

In [ ]:
# FULL dataset (~42GB, ~20-40 min). Fits on Colab's ephemeral disk.
!kaggle datasets download -d nih-chest-xrays/data -p /content/data --unzip

# --- FASTER ALTERNATIVE (uncomment, comment the line above) ---
# Sample subset (~5k images, much quicker, but poor coverage of rare labels):
# !kaggle datasets download -d nih-chest-xrays/sample -p /content/data --unzip

!ls /content/data | head

## 5. Mount Drive for checkpoints (survives disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/chestxray_outputs

## 6. Configure for the free tier
Edits `config.py`: point at the data, subsample by patient, smaller batches,
fewer epochs, and write outputs to Drive.

In [ ]:
import re, pathlib
cfg = pathlib.Path('config.py').read_text()
edits = {
    r'DATA_DIR = .*': 'DATA_DIR = Path("/content/data")',
    r'OUTPUT_DIR = .*': 'OUTPUT_DIR = Path("/content/drive/MyDrive/chestxray_outputs")',
    r'BATCH_SIZE = .*': 'BATCH_SIZE = 32',
    r'NUM_WORKERS = .*': 'NUM_WORKERS = 2',
    r'EPOCHS = .*': 'EPOCHS = 6',
    r'MAX_TRAIN_PATIENTS = .*': 'MAX_TRAIN_PATIENTS = 3000   # ~10k images',
    r'MAX_EVAL_IMAGES = .*': 'MAX_EVAL_IMAGES = 4000',
}
for pat, rep in edits.items():
    cfg = re.sub(pat, rep, cfg, count=1)
pathlib.Path('config.py').write_text(cfg)
print('config.py updated. Set MAX_TRAIN_PATIENTS=None later for the full run.')

## 7. Sanity check (no training yet)

In [ ]:
!python run_smoke_test.py

## 8. Train the three models
VGG-19 is the slow one. If it OOMs, lower BATCH_SIZE to 16 in cell 6 and rerun.

In [ ]:
!python -m src.train --model resnet18

In [ ]:
!python -m src.train --model vgg19

In [ ]:
!python -m src.train --model customcnn

## 9. Comparison tables + figures

In [ ]:
!python -m src.report_figures
from IPython.display import Image, display
import os
fig = '/content/drive/MyDrive/chestxray_outputs/figures'
for f in ['learning_curves.png', 'auroc_comparison.png']:
    if os.path.exists(f'{fig}/{f}'):
        display(Image(f'{fig}/{f}'))

## 10. Grad-CAM on a sample image

In [ ]:
import glob
sample = sorted(glob.glob('/content/data/**/*.png', recursive=True))[0]
!python -m src.gradcam --model resnet18 --image "{sample}" --label Cardiomegaly

## 11. RAG index + grounded LLM summary (Gemini)

In [ ]:
import os
os.environ['GEMINI_API_KEY'] = 'PASTE_YOUR_GEMINI_KEY'   # aistudio.google.com
!python -m src.rag.build_index
!python -m src.rag.interpret --demo

## Done
Everything is in `MyDrive/chestxray_outputs` (checkpoints, metrics, figures).
Paste the numbers into `report/report_template.md`, then push to GitHub.

**To scale up to the full dataset later:** in cell 6 set `MAX_TRAIN_PATIENTS = None`,
`MAX_EVAL_IMAGES = None`, `EPOCHS = 10`, and rerun (expect a much longer run that
may need more than one session — the Drive checkpoints let you resume per model).